# 06 · Every number in the manuscript, and Table 2

All metrics are recomputed from the per-patch `predictions.csv` files — never read from a
summary — and compared with `reference/manuscript_numbers.json`, the numbers the
manuscript text, tables and figures were written from. Values are compared at the three
decimals the manuscript prints.

* Binary rat tasks: F1 for the more injured class (VML; Very poor). Three-class: macro-F1.
* WSI level: patch probabilities averaged per slide, then argmax.
* ROC-AUC: binary — the more injured class; three-class — macro one-vs-rest.

This notebook needs no images: it reads the saved predictions in `models/`. `RUN = True`
reads a re-run of the original data in `runs/rerun` instead.

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
SYNTHETIC = False
os.environ.pop("VML_SYNTHETIC", None)   # always the manuscript's saved predictions
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
print("results (used) :", P.RESULTS)

In [ ]:
RUN = SYNTHETIC                           # <- True: train into RERUN_BASE (always on synthetic data)
RERUN_BASE = P.WORK / "rerun"
BASE = RERUN_BASE if RUN else P.RESULTS   # where this notebook reads results from

In [ ]:
from manuscript_numbers import compute, compare
N = compute(BASE)
P.WORK.mkdir(parents=True, exist_ok=True)
(P.WORK / "manuscript_numbers.json").write_text(json.dumps(N, indent=1))
cmp = compare(N)
print(f"{(cmp.status == 'match').sum()} of {len(cmp)} manuscript numbers reproduced to 3 decimals")
cmp[cmp.status != "match"]

## Rat results (Figures 1–3)

In [ ]:
R = pd.DataFrame(N["rat"]).T
R.index = pd.MultiIndex.from_tuples([tuple(k.split("__")) for k in R.index], names=["model", "task"])
R = R.swaplevel().sort_index()
f1 = lambda lvl: [R.loc[(t, m), f"{lvl}_macro" if t == "multiclass" else f"{lvl}_f1_pos"] for t, m in R.index]
R.insert(0, "patch_F1", f1("patch")); R.insert(1, "wsi_F1", f1("wsi"))
R[["patch_F1", "wsi_F1", "auc", "patch_acc", "wsi_acc", "n_patch", "n_wsi"]].astype(float).round(3)

## Table 2 — ResNet-50 ablation (image only / morphometrics only / fusion)

In [ ]:
rows = []
for t, name in [("control_vml", "Control vs VML"), ("poor_very_poor", "Poor vs Very poor"), ("multiclass", "Three-class (macro)")]:
    col = lambda lvl: f"{lvl}_macro" if t == "multiclass" else f"{lvl}_f1_pos"
    v = lambda m, lvl: f"{N['rat'][f'{m}__{t}'][col(lvl)]:.3f}"
    rows.append({"task": name,
                 "patch: image": v("cnn", "patch"), "patch: morph": v("mlp_morph", "patch"), "patch: fusion": v("cnn_morph", "patch"),
                 "WSI: image": v("cnn", "wsi"), "WSI: morph": v("mlp_morph", "wsi"), "WSI: fusion": v("cnn_morph", "wsi")})
table2 = pd.DataFrame(rows).set_index("task")
table2.to_csv(P.WORK / "table2_ablation.csv")
table2

## Human transfer (Figure 4)

In [ ]:
pd.DataFrame(N["human"]).T[["patch_f1", "wsi_f1", "patch_macro", "wsi_macro", "auc"]].astype(float).round(3)